# Thư viện

In [203]:
# !pip install python-docx textract


In [204]:
import os
import re
import pandas as pd
import docx
import textract

In [205]:


import enum


class DocumentMetadata:
    """Class đại diện cho thông tin cấu trúc (Metadata) của một văn bản"""
    def __init__(self, 
                 file_path:str, 
                 doc_type: str=None, 
                 issued_by: str=None, 
                 law_name: str=None, 
                 code_number: str=None
    ):
        self.file_path = file_path
        self.file_name = os.path.basename(file_path)
        self.doc_type = doc_type       # Ví dụ: Hiến pháp, Luật, Nghị quyết...
        self.issued_by = issued_by     # Cơ quan ban hành (Ví dụ: Quốc hội)
        self.law_name = law_name       # Tên văn bản/luật
        self.code_number = code_number # Số hiệu văn bản (Ví dụ: 91/2015/QH13)
        self.cleaned_content = ""      # Nội dung văn bản sau khi chuẩn hóa

    def to_dict(self):
        """Chuyển đổi đối tượng thành dictionary để dễ tạo DataFrame/Lưu DB"""
        return {
            "ten_file": self.file_name,
            "duong_dan_file": self.file_path,
            "loai_van_ban": self.doc_type,
            "co_quan_ban_hanh": self.issued_by,
            "ten_van_ban": self.law_name,
            "so_hieu": self.code_number,
            "noi_dung_chuan_hoa": self.cleaned_content
        }

class FileType(enum.Enum):
    DOCX = '.docx'
    DOC = '.doc'


class DocumentParser:
    """Class phụ trách đọc file word và chuẩn hóa nội dung văn bản"""
    @staticmethod
    def read_file(file_path):
        # 1. Kiểm tra file có thực sự tồn tại hay không
        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"Không tìm thấy file tại đường dẫn: {os.path.abspath(file_path)}\n"
                f"Vui lòng kiểm tra lại tên thư mục (gạch ngang '-' hay gạch dưới '_') và tên file."
            )
            
        ext = os.path.splitext(file_path)[1].lower()
        print(f"Đang đọc file: {file_path} (Định dạng: {ext[1]})")
        if ext == FileType.DOCX.value:
            doc = docx.Document(file_path)
            return '\n'.join([para.text for para in doc.paragraphs])
        elif ext == FileType.DOC.value:
            return textract.process(file_path).decode('utf-8')
        else:
            raise ValueError(f"Định dạng file {ext} không được hỗ trợ.")

    @staticmethod
    def clean_text(text):
        if not text:
            return ""
        text = re.sub(r'\r\n|\r', '\n', text)
        lines = [line.strip() for line in text.split('\n')]
        cleaned_text = '\n'.join([line for line in lines if line != ""])
        cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)
        return cleaned_text

    def parse(self, doc_obj: DocumentMetadata):
        raw_text = self.read_file(doc_obj.file_path)
        doc_obj.cleaned_content = self.clean_text(raw_text)
        return doc_obj


class DataExporter:
    """Class phụ trách xuất dữ liệu ra CSV/Excel hoặc tích hợp kết nối Database"""
    def __init__(self, documents_list: list[DocumentMetadata]):
        self.documents = documents_list

    def to_dataframe(self):
        """Chuyển đổi danh sách đối tượng thành Pandas DataFrame"""
        data = [doc.to_dict() for doc in self.documents]
        return pd.DataFrame(data)

    def export_to_csv(self, output_path):
        """Xuất dữ liệu ra file CSV"""
        df = self.to_dataframe()
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f" Đã xuất dữ liệu thành công ra file CSV tại: {output_path}")

    def export_to_sql(self, table_name, db_engine):
        """Xuất dữ liệu thẳng vào Database (Sử dụng SQLAlchemy engine)"""
        df = self.to_dataframe()
        # Lưu vào DB, nếu bảng đã tồn tại thì chèn thêm dữ liệu (append)
        df.to_sql(table_name, con=db_engine, if_exists='append', index=False)
        print(f" Đã lưu thành công {len(df)} dòng vào bảng '{table_name}' trong Database.")


In [206]:
class Config:
    """Class chứa các cấu hình và hằng số cho quá trình xử lý dữ liệu"""
    DATA_DIR = '../data/van_ban'
    METADATA_FILE = '../data/metadata/file_metadata.csv'
    OUTPUT_FILE = '../data/van_ban_chuan_hoa.csv'
    
    

In [207]:
# 1. Đọc file metadata quản lý từ trước
meta = [
    {
        "file_path": "../data/van_ban/Hien_phap_nam_2013.docx",
        "type": "Hiến pháp",
        "do_ai_ban_hanh": "Quốc hội",
        "ten_luat": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013",
        "so_hieu": "Số 15/2013/QH11"
    },
    {
        "file_path": "../data/van_ban/bo_luat_dan_su_91-2015-QH13.docx",
        "type": "Luật",
        "do_ai_ban_hanh": "Quốc hội",
        "ten_luat": "Luật Dân sự",
        "so_hieu": "Số 91/2015/QH13"
    }
]
df_meta = pd.DataFrame(meta)
 

df_meta.to_csv(Config.METADATA_FILE, index=False, encoding='utf-8-sig')

if not os.path.exists(Config.METADATA_FILE):
    raise FileNotFoundError(f"Không tìm thấy file metadata tại: {Config.METADATA_FILE}")
df_meta = pd.read_csv(Config.METADATA_FILE) 

documents_pool: list[DocumentMetadata] = []
parser = DocumentParser()

# 2. Áp dụng OOP để khởi tạo đối tượng và xử lý text
for index, row in df_meta.iterrows():
    # Khởi tạo đối tượng từ thông tin có sẵn trong CSV/DB
    doc = DocumentMetadata(
        file_path=row['file_path'],
        doc_type=row['type'],
        issued_by=row['do_ai_ban_hanh'],
        law_name=row['ten_luat'],
        code_number=row.get('so_hieu', None)
    )
    
    try:
        # Tiến hành đọc và parse nội dung text vào object
        parser.parse(doc)
        documents_pool.append(doc)
        print(f"Xử lý xong: {doc.file_name}")
    except Exception as e:
        print(f"Lỗi xử lý file {row['file_path']}: {e}")
        
if (len(documents_pool) == 0):
    print("Không có văn bản nào được xử lý thành công. Vui lòng kiểm tra lại file metadata hoặc đường dẫn file.")
else:
    # 3. Xuất kết quả cuối cùng chứa cả metadata lẫn nội dung đã chuẩn hóa
    exporter = DataExporter(documents_pool)
    exporter.export_to_csv(Config.OUTPUT_FILE)


Đang đọc file: ../data/van_ban/Hien_phap_nam_2013.docx (Định dạng: d)
Xử lý xong: Hien_phap_nam_2013.docx
Đang đọc file: ../data/van_ban/bo_luat_dan_su_91-2015-QH13.docx (Định dạng: d)
Xử lý xong: bo_luat_dan_su_91-2015-QH13.docx
 Đã xuất dữ liệu thành công ra file CSV tại: ../data/van_ban_chuan_hoa.csv


In [208]:
df = pd.read_csv(Config.OUTPUT_FILE)

display(df.head(5))
display(df.info())

,ten_file,duong_dan_file,loai_van_ban,co_quan_ban_hanh,ten_van_ban,so_hieu,noi_dung_chuan_hoa
0,Hien_phap_nam_2013.docx,../data/van_ban/Hien_phap_nam_2013.docx,Hiến pháp,Quốc hội,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt ...,Số 15/2013/QH11,HIẾN PHÁP\nNƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...
1,bo_luat_dan_su_91-2015-QH13.docx,../data/van_ban/bo_luat_dan_su_91-2015-QH13.docx,Luật,Quốc hội,Luật Dân sự,Số 91/2015/QH13,BỘ LUẬT\nDÂN SỰ\nCăn cứ Hiến pháp nước Cộng hò...


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ten_file            2 non-null      str  
 1   duong_dan_file      2 non-null      str  
 2   loai_van_ban        2 non-null      str  
 3   co_quan_ban_hanh    2 non-null      str  
 4   ten_van_ban         2 non-null      str  
 5   so_hieu             2 non-null      str  
 6   noi_dung_chuan_hoa  2 non-null      str  
dtypes: str(7)
memory usage: 568.3 KB


None

In [209]:
for index, row in df.iterrows():
    print("=" * 80)
    text = row['noi_dung_chuan_hoa']
    print(f"Văn bản {index+1}: {row['ten_file']}")
    print(f"Nội dung chuẩn hóa:\n{text[:500]}...")
    print("=" * 80)

Văn bản 1: Hien_phap_nam_2013.docx
Nội dung chuẩn hóa:
HIẾN PHÁP
NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT
LỜI NÓI ĐẦU
Trải qua mấy nghìn năm lịch sử, Nhân dân Việt Nam lao động cần cù, sáng tạo, đấu tranh anh dũng để dựng nước và giữ nước, đã hun đúc nên truyền thống yêu nước, đoàn kết, nhân nghĩa, kiên cường, bất khuất và xây dựng nên nền văn hiến Việt Nam.
Từ năm 1930, dưới sự lãnh đạo của Đảng Cộng sản Việt Nam do Chủ tịch Hồ Chí Minh sáng lập và rèn luyện, Nhân dân ta tiến hành cuộc đấu tranh lâu dài, đầy gian khổ, hy sinh vì độc l...
Văn bản 2: bo_luat_dan_su_91-2015-QH13.docx
Nội dung chuẩn hóa:
BỘ LUẬT
DÂN SỰ
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;
Quốc hội ban hành Bộ luật dân sự.
Phần thứ nhất
QUY ĐỊNH CHUNG
Chương I
NHỮNG QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh
Bộ luật này quy định địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử của cá nhân, pháp nhân; quyền, nghĩa vụ về nhân thân và tài sản của cá nhân, pháp nhân trong các quan hệ được hình

# Đường dẫn và cấu hình

# Lớp nhận data và chuyển data thành text thuần (docx, pdf, ...)
- Hiện tại hỗ trợ docx
- File data có thể ở: thư mục làm việc, storages(S3, Minio)

# Lớp nhận dạng text theo regex và đưa text thành data cấu trúc
- Hỗ trợ các văn bản: Hiện pháp, Bộ Luật, Nghị định, 

## models.py

In [210]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional


@dataclass
class Diem:
    so: str
    noi_dung: str = ""

    def to_dict(self) -> dict:
        return {
            "loai": "diem",
            "so": self.so,
            "noi_dung": self.noi_dung,
        }


@dataclass
class Khoan:
    so: int
    noi_dung: str = ""
    cac_diem: list[Diem] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "khoan",
            "so": self.so,
            "noi_dung": self.noi_dung if self.noi_dung else None,
            "cac_diem": [
                diem.to_dict()
                for diem in self.cac_diem
            ],
        }


@dataclass
class Dieu:
    so: int
    tieu_de: Optional[str] = None
    noi_dung: str = ""
    cac_khoan: list[Khoan] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "dieu",
            "so": self.so,
            "tieu_de": self.tieu_de,
            "noi_dung": self.noi_dung if not self.cac_khoan else None,
            "cac_khoan": [
                khoan.to_dict()
                for khoan in self.cac_khoan
            ],
        }


@dataclass
class TieuMuc:
    so: str
    ten: str
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "tieu_muc",
            "so": self.so,
            "ten": self.ten,
            "cac_dieu": [
                dieu.to_dict()
                for dieu in self.cac_dieu
            ],
        }


@dataclass
class Muc:
    so: str
    ten: str
    cac_tieu_muc: list[TieuMuc] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "muc",
            "so": self.so,
            "ten": self.ten,
            "cac_tieu_muc": [
                item.to_dict()
                for item in self.cac_tieu_muc
            ],
            "cac_dieu": [
                dieu.to_dict()
                for dieu in self.cac_dieu
            ],
        }


@dataclass
class Chuong:
    so: str
    ten: str
    cac_muc: list[Muc] = field(default_factory=list)
    cac_tieu_muc: list[TieuMuc] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "chuong",
            "so": self.so,
            "ten": self.ten,
            "cac_muc": [
                muc.to_dict()
                for muc in self.cac_muc
            ],
            "cac_tieu_muc": [
                item.to_dict()
                for item in self.cac_tieu_muc
            ],
            "cac_dieu": [
                dieu.to_dict()
                for dieu in self.cac_dieu
            ],
        }


@dataclass
class Phan:
    so: str
    ten: str
    cac_chuong: list[Chuong] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "phan",
            "so": self.so,
            "ten": self.ten,
            "cac_chuong": [
                chuong.to_dict()
                for chuong in self.cac_chuong
            ],
            "cac_dieu": [
                dieu.to_dict()
                for dieu in self.cac_dieu
            ],
        }


@dataclass
class VanBan:
    ten: str
    loai_van_ban: Optional[str] = None
    so_van_ban: Optional[str] = None

    # Nội dung trước khi bắt đầu Phần/Chương/Điều
    phan_mo_dau: list[str] = field(default_factory=list)

    cac_phan: list[Phan] = field(default_factory=list)
    cac_chuong: list[Chuong] = field(default_factory=list)
    cac_dieu: list[Dieu] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "loai": "van_ban",
            "ten": self.ten,
            "loai_van_ban": self.loai_van_ban,
            "so_van_ban": self.so_van_ban,
            "phan_mo_dau": self.phan_mo_dau,
            "cac_phan": [
                phan.to_dict()
                for phan in self.cac_phan
            ],
            "cac_chuong": [
                chuong.to_dict()
                for chuong in self.cac_chuong
            ],
            "cac_dieu": [
                dieu.to_dict()
                for dieu in self.cac_dieu
            ],
        }


## patterns.py

In [211]:
import re


# =========================
# PHẦN
# =========================
PATTERN_PHAN = re.compile(
    r"^Phần\s+(?P<so>.+?)(?:\.\s*(?P<ten>.*))?$",
    re.IGNORECASE
)

# =========================
# CHƯƠNG
# =========================

PATTERN_CHUONG = re.compile(
    r"^Chương\s+(?P<so>[IVXLCDM]+)"
    r"(?:\.\s*(?P<ten>.*))?$",
    re.IGNORECASE
)

# =========================
# MỤC
# =========================

PATTERN_MUC = re.compile(
    r"^Mục\s+(?P<so>\d+)"
    r"(?:\.\s*(?P<ten>.*))?$",
    re.IGNORECASE
)


# =========================
# TIỂU MỤC
# =========================

PATTERN_TIEU_MUC = re.compile(
    r"^Tiểu mục\s+(?P<so>\d+)"
    r"(?:\.\s*(?P<ten>.*))?$",
    re.IGNORECASE
)


# =========================
# ĐIỀU
# =========================

PATTERN_DIEU = re.compile(
    r"^Điều\s+(?P<so>\d+)"
    r"(?:\.\s*(?P<tieu_de>.*))?$",
    re.IGNORECASE
)


# =========================
# KHOẢN
# =========================

PATTERN_KHOAN = re.compile(
    r"^(?P<so>\d+)\.\s+(?P<noi_dung>.+)$"
)


# =========================
# ĐIỂM
# =========================

PATTERN_DIEM = re.compile(
    r"^(?P<so>[a-zđ])\)\s+(?P<noi_dung>.+)$",
    re.IGNORECASE
)

## parser.py

In [ ]:
from __future__ import annotations

from typing import Optional

# from models import (
#     VanBan,
#     Phan,
#     Chuong,
#     Muc,
#     TieuMuc,
#     Dieu,
#     Khoan,
#     Diem,
# )

# from patterns import (
#     PATTERN_PHAN,
#     PATTERN_CHUONG,
#     PATTERN_MUC,
#     PATTERN_TIEU_MUC,
#     PATTERN_DIEU,
#     PATTERN_KHOAN,
#     PATTERN_DIEM,
# )

# from cleaner import chuan_hoa_text


class ParserVanBanPhapLuat:

    def __init__(
        self,
        ten_van_ban: str,
        loai_van_ban: Optional[str] = None,
        so_van_ban: Optional[str] = None,
    ):
        self.van_ban = VanBan(
            ten=ten_van_ban,
            loai_van_ban=loai_van_ban,
            so_van_ban=so_van_ban,
        )

        # Context hiện tại
        self.phan_hien_tai: Optional[Phan] = None
        self.chuong_hien_tai: Optional[Chuong] = None
        self.muc_hien_tai: Optional[Muc] = None
        self.tieu_muc_hien_tai: Optional[TieuMuc] = None
        self.dieu_hien_tai: Optional[Dieu] = None
        self.khoan_hien_tai: Optional[Khoan] = None
        self.diem_hien_tai: Optional[Diem] = None

    # def parse(self, text: str) -> VanBan:

    #     lines = chuan_hoa_text(text)

    #     for line in lines:
    #         self._xu_ly_dong(line)

    #     return self.van_ban
    
    def parse(self, text: str) -> VanBan:

        lines = chuan_hoa_text(text)

        i = 0
        
        waiting_for_title = True 

        while i < len(lines):

            line = lines[i]
            
            # Đối với chương và phần thì title là dòng tiếp theo
            if waiting_for_title:
                if self.chuong_hien_tai is not None and self.chuong_hien_tai.ten == "":
                    self.chuong_hien_tai.ten = line.strip()
                    waiting_for_title = False
                    i += 1
                    continue
                elif self.phan_hien_tai is not None and self.phan_hien_tai.ten == "":
                    self.phan_hien_tai.ten = line.strip()
                    waiting_for_title = False
                    i += 1
                    continue

            # ==================================================
            # PHẦN
            # ==================================================

            match = PATTERN_PHAN.match(line)

            if match:

                self._tao_phan(
                    so=match.group("so"),
                    ten=match.group("ten"),
                )

                i += 1
                continue

            # ==================================================
            # CHƯƠNG
            # ==================================================

            match = PATTERN_CHUONG.match(line)

            if match:

                self._tao_chuong(
                    so=match.group("so"),
                    ten=match.group("ten"),
                )

                i += 1
                waiting_for_title = True 
                continue

            # ==================================================
            # MỤC
            # ==================================================

            match = PATTERN_MUC.match(line)

            if match:
                self._tao_muc(
                    so=match.group("so"),
                    ten=match.group("ten"),
                )

                i += 1
                continue

            # ==================================================
            # TIỂU MỤC
            # ==================================================

            match = PATTERN_TIEU_MUC.match(line)

            if match:

                self._tao_tieu_muc(
                    so=match.group("so"),
                    ten=match.group("ten"),
                )

                i += 1
                continue

            # ==================================================
            # ĐIỀU
            # ==================================================

            match = PATTERN_DIEU.match(line)

            if match:

                self._tao_dieu(
                    so=int(match.group("so")),
                    tieu_de=match.group("tieu_de"),
                )

                i += 1
                continue

            # ==================================================
            # KHOẢN
            # ==================================================

            match = PATTERN_KHOAN.match(line)

            if match and self.dieu_hien_tai is not None:

                self._tao_khoan(
                    so=int(match.group("so")),
                    noi_dung=match.group("noi_dung"),
                )

                i += 1
                continue

            # ==================================================
            # ĐIỂM
            # ==================================================

            match = PATTERN_DIEM.match(line)

            if match and self.khoan_hien_tai is not None:

                self._tao_diem(
                    so=match.group("so"),
                    noi_dung=match.group("noi_dung"),
                )

                i += 1
                continue

            # ==================================================
            # NỘI DUNG
            # ==================================================

            self._them_noi_dung(line)

            i += 1

        return self.van_ban
    
    def _co_the_la_tieu_de(self, line: str) -> bool:
        

        if not line:
            return False

        # Không phải Điều
        if PATTERN_DIEU.match(line):
            return False

        # Không phải Khoản
        if PATTERN_KHOAN.match(line):
            return False

        # Không phải Điểm
        if PATTERN_DIEM.match(line):
            return False

        # Không phải cấp cấu trúc khác
        if PATTERN_PHAN.match(line):
            return False

        if PATTERN_CHUONG.match(line):
            return False

        if PATTERN_MUC.match(line):
            return False

        if PATTERN_TIEU_MUC.match(line):
            return False

        return True

    # =====================================================
    # XỬ LÝ TỪNG DÒNG
    # =====================================================

    def _xu_ly_dong(self, line: str):

        # -------------------------
        # PHẦN
        # -------------------------

        match = PATTERN_PHAN.match(line)

        if match:
            self._tao_phan(
                so=match.group("so"),
                ten=None,
            )
            return

        # -------------------------
        # CHƯƠNG
        # -------------------------

        match = PATTERN_CHUONG.match(line)

        if match:
            self._tao_chuong(
                so=match.group("so"),
                ten=None,
            )
            return

        # -------------------------
        # MỤC
        # -------------------------

        match = PATTERN_MUC.match(line)

        if match:
            self._tao_muc(
                so=match.group("so"),
                ten=None,
            )
            return

        # -------------------------
        # TIỂU MỤC
        # -------------------------

        match = PATTERN_TIEU_MUC.match(line)

        if match:
            self._tao_tieu_muc(
                so=match.group("so"),
                ten=None,
            )
            return

        # -------------------------
        # ĐIỀU
        # -------------------------

        match = PATTERN_DIEU.match(line)

        if match:
            self._tao_dieu(
                so=int(match.group("so")),
                tieu_de=match.group("tieu_de"),
            )
            return

        # -------------------------
        # KHOẢN
        # -------------------------

        match = PATTERN_KHOAN.match(line)

        if match and self.dieu_hien_tai is not None:

            self._tao_khoan(
                so=int(match.group("so")),
                noi_dung=match.group("noi_dung"),
            )
            return

        # -------------------------
        # ĐIỂM
        # -------------------------

        match = PATTERN_DIEM.match(line)

        if match and self.khoan_hien_tai is not None:

            self._tao_diem(
                so=match.group("so"),
                noi_dung=match.group("noi_dung"),
            )
            return

        # -------------------------
        # NỘI DUNG THƯỜNG
        # -------------------------

        self._them_noi_dung(line)

    # =====================================================
    # TẠO PHẦN
    # =====================================================

    def _tao_phan(
        self,
        so: str,
        ten: Optional[str],
    ):

        phan = Phan(
            so=so,
            ten=ten or "",
        )

        self.van_ban.cac_phan.append(phan)

        self.phan_hien_tai = phan

        self.chuong_hien_tai = None
        self.muc_hien_tai = None
        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    # =====================================================
    # TẠO CHƯƠNG
    # =====================================================

    def _tao_chuong(
        self,
        so: str,
        ten: Optional[str],
    ):

        chuong = Chuong(
            so=so,
            ten=ten.strip() if ten else "",
        )

        if self.phan_hien_tai is not None:
            self.phan_hien_tai.cac_chuong.append(chuong)
        else:
            self.van_ban.cac_chuong.append(chuong)

        self.chuong_hien_tai = chuong

        self.muc_hien_tai = None
        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None
    # =====================================================
    # TẠO MỤC
    # =====================================================
    
    def _tao_muc(
        self,
        so: str,
        ten: Optional[str],
    ):

        if self.chuong_hien_tai is None:
            raise ValueError(
                f"Mục {so} xuất hiện nhưng chưa có Chương."
            )

        muc = Muc(
            so=so,
            ten=ten.strip() if ten else "",
        )

        self.chuong_hien_tai.cac_muc.append(muc)

        self.muc_hien_tai = muc

        self.tieu_muc_hien_tai = None
        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None
    # =====================================================
    # TẠO TIỂU MỤC
    # =====================================================

    def _tao_tieu_muc(
        self,
        so: str,
        ten: Optional[str],
    ):

        tieu_muc = TieuMuc(
            so=so,
            ten=ten.strip() if ten else "",
        )

        if self.muc_hien_tai is not None:

            self.muc_hien_tai.cac_tieu_muc.append(tieu_muc)

        elif self.chuong_hien_tai is not None:

            self.chuong_hien_tai.cac_tieu_muc.append(tieu_muc)

        else:

            raise ValueError(
                f"Tiểu mục {so} xuất hiện nhưng "
                f"chưa có Mục/Chương."
            )

        self.tieu_muc_hien_tai = tieu_muc

        self.dieu_hien_tai = None
        self.khoan_hien_tai = None
        self.diem_hien_tai = None
    # =====================================================
    # TẠO ĐIỀU
    # =====================================================

    def _tao_dieu(
        self,
        so: int,
        tieu_de: Optional[str],
    ):

        dieu = Dieu(
            so=so,
            tieu_de=tieu_de.strip()
            if tieu_de
            else None,
        )

        # Ưu tiên nơi sâu nhất hiện tại
        if self.tieu_muc_hien_tai is not None:

            self.tieu_muc_hien_tai.cac_dieu.append(dieu)

        elif self.muc_hien_tai is not None:

            self.muc_hien_tai.cac_dieu.append(dieu)

        elif self.chuong_hien_tai is not None:

            self.chuong_hien_tai.cac_dieu.append(dieu)

        elif self.phan_hien_tai is not None:

            self.phan_hien_tai.cac_dieu.append(dieu)

        else:

            self.van_ban.cac_dieu.append(dieu)

        self.dieu_hien_tai = dieu

        self.khoan_hien_tai = None
        self.diem_hien_tai = None

    # =====================================================
    # TẠO KHOẢN
    # =====================================================

    def _tao_khoan(
        self,
        so: int,
        noi_dung: str,
    ):

        if self.dieu_hien_tai is None:
            return

        khoan = Khoan(
            so=so,
            noi_dung=noi_dung,
        )

        self.dieu_hien_tai.cac_khoan.append(khoan)

        self.khoan_hien_tai = khoan
        self.diem_hien_tai = None

    # =====================================================
    # TẠO ĐIỂM
    # =====================================================

    def _tao_diem(
        self,
        so: str,
        noi_dung: str,
    ):

        if self.khoan_hien_tai is None:
            return

        diem = Diem(
            so=so,
            noi_dung=noi_dung,
        )

        self.khoan_hien_tai.cac_diem.append(diem)

        self.diem_hien_tai = diem

    # =====================================================
    # NỘI DUNG
    # =====================================================

    def _them_noi_dung(self, line: str):

        # -----------------------------------------
        # Chưa vào Điều
        # -----------------------------------------

        if self.dieu_hien_tai is None:
            
            if self.chuong_hien_tai is not None:
                self.chuong_hien_tai.ten += " " + line
            elif self.phan_hien_tai is not None:
                self.phan_hien_tai.ten += " " + line            
            else:
                self.van_ban.phan_mo_dau.append(line)

            return

        # -----------------------------------------
        # Đang ở Điểm
        # -----------------------------------------

        if self.diem_hien_tai is not None:

            self.diem_hien_tai.noi_dung += " " + line

            return

        # -----------------------------------------
        # Đang ở Khoản
        # -----------------------------------------

        if self.khoan_hien_tai is not None:

            self.khoan_hien_tai.noi_dung += " " + line

            return

        # -----------------------------------------
        # Điều không có Khoản
        # -----------------------------------------

        self.dieu_hien_tai.noi_dung += " " + line

## cleaners.py

In [213]:
import re
import unicodedata


def chuan_hoa_dong(line: str) -> str:
    """
    Chuẩn hóa một dòng:
    - Xóa BOM
    - Chuẩn hóa Unicode về NFC
    - Chuẩn hóa khoảng trắng
    - Loại bỏ khoảng trắng đầu/cuối
    """

    if not line:
        return ""

    # Xóa BOM
    line = line.replace("\ufeff", "")

    # Quan trọng:
    # "Điều" -> "Điều"
    line = unicodedata.normalize("NFC", line)

    # Chuẩn hóa whitespace
    line = re.sub(r"[ \t]+", " ", line)

    return line.strip()


def chuan_hoa_text(text: str) -> list[str]:
    """
    Text -> danh sách dòng sạch.
    """

    if not text:
        return []

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    lines = []

    for raw_line in text.split("\n"):
        line = chuan_hoa_dong(raw_line)

        if line:
            lines.append(line)

    return lines

## csv_reader.py

In [214]:
import csv
import pandas as pd

# from parser import ParserVanBanPhapLuat


def parse_csv(path: str) -> list[dict]:

    results = []
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file CSV tại: {path}")
    
    df = pd.read_csv(path, encoding="utf-8-sig")
    
    for index, row in df.iterrows():
        text = row.get("noi_dung_chuan_hoa", "")

        if not text.strip():
            continue

        parser = ParserVanBanPhapLuat(
            ten_van_ban=row.get("ten_van_ban") or "Không rõ",
            loai_van_ban=row.get("loai_van_ban"),
            so_van_ban=row.get("so_van_ban"),
        )

        van_ban = parser.parse(text)

        results.append({
            "metadata": {
                key: value
                for key, value in row.items()
                if key != "noi_dung_chuan_hoa"
            },
            "cau_truc": van_ban.to_dict()
        })

    return results

## main.py

In [215]:
import json

# from csv_reader import parse_csv


def main():

    results = parse_csv(
        Config.OUTPUT_FILE
    )
    
    print(f"Đã phân tích xong {len(results)} văn bản pháp luật.")
    
    json_output_path = os.path.join(
        os.path.dirname(Config.DATA_DIR),
        "du_lieu_phap_luat.json"
    )
    

    with open(
        json_output_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            results,
            file,
            ensure_ascii=False,
            indent=2
        )


if __name__ == "__main__":
    main()

Đã phân tích xong 2 văn bản pháp luật.
